# 01 — Data Understanding and Exploration

Document the dataset context, structure, quality, distributions, and initial analytical findings before data preparation and modeling.

## 1. Study Context

Customer churn occurs when a customer ends their relationship with a service provider. In the telecommunications context, identifying patterns associated with churn can support more focused customer-retention actions.

This study uses the Telco Customer Churn dataset to investigate how customer characteristics, subscribed services, contract conditions, tenure, and charges relate to the `Churn` outcome. The exploratory stage will evaluate the dataset's structure and quality, identify relevant behavioral patterns, and establish evidence for the later development and comparison of binary classification models.

The dataset source summarizes the business objective as:

> “Predict behavior to retain customers. You can analyze all relevant customer data and develop focused customer retention programs.” — IBM Sample Data Sets

The study is guided by the following questions:

* Which customer, service, and account characteristics are most associated with churn?
* Are there data-quality issues or class imbalance that may affect the analysis?
* Which findings should guide data preparation, feature treatment, and model evaluation?

## 2. Dataset Source

The study uses the **Telco Customer Churn** dataset published on Kaggle by **BlastChar** and attributed to **IBM Sample Data Sets**.

| Item                     | Value                                                                                                          |
| ------------------------ | -------------------------------------------------------------------------------------------------------------- |
| Dataset                  | Telco Customer Churn                                                                                           |
| Platform                 | Kaggle                                                                                                         |
| Kaggle handle            | `blastchar/telco-customer-churn`                                                                               |
| Original attribution     | IBM Sample Data Sets                                                                                           |
| Acquisition method       | `kagglehub`, through `scripts/download_data.py`                                                                |
| Local raw-data directory | `data/raw/telco-customer-churn/`                                                                               |
| Retrieved version        | Latest available version at execution time                                                                     |
| Usage note               | Preserve the original source attribution and review the source usage conditions before redistributing the data |

The downloaded files are treated as immutable source data. Cleaning, type correction, filtering, and other transformations must not overwrite files under `data/raw/`. Derived datasets should be written to `data/interim/` or `data/processed/`.

In [1]:
# Import the utility that locates and validates the project root.
from scripts.project_context import get_project_context


# Local identifier used by this study's directories and artifacts.
DATASET_SLUG = "telco-customer-churn"

# Initialize a cross-platform project context.
PROJECT = get_project_context()

# Build the raw-data path from the dataset-specific slug.
RAW_DATA_DIR = PROJECT.path(
    "data",
    "raw",
    DATASET_SLUG,
)

# Display only safe project-relative information.
print(f"Project: {PROJECT.name}")
print(f"Raw data directory: {PROJECT.display(RAW_DATA_DIR)}")


Project: dataset-study-telco-customer-churn
Raw data directory: data/raw/telco-customer-churn


In [2]:
# Import the Kaggle acquisition operation.
from scripts.download_data import acquire_kaggle_dataset


# Public Kaggle identifier for this study's dataset.
DATASET_HANDLE = "blastchar/telco-customer-churn"

# Download the dataset or reuse its existing local materialization.
# Third-party progress output is disabled to avoid exposing absolute paths.
acquisition = acquire_kaggle_dataset(
    handle=DATASET_HANDLE,
    destination=RAW_DATA_DIR,
    project_root=PROJECT.root,
    show_progress=False,
)

print(
    "Dataset source: "
    f"Kaggle — {acquisition.source_reference}"
)
print(
    "Raw data directory: "
    f"{PROJECT.display(acquisition.destination)}"
)
print(f"Dataset files: {len(acquisition.files)}")

for file_path in acquisition.files:
    print(f"- {file_path.name}")

Dataset source: Kaggle — blastchar/telco-customer-churn
Raw data directory: data/raw/telco-customer-churn
Dataset files: 1
- WA_Fn-UseC_-Telco-Customer-Churn.csv


In [3]:
import pandas as pd


# Main source file selected for this study.
DATASET_FILE_SELECTOR = (
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# Keep parsing choices visible because they affect interpretation.
CSV_READ_OPTIONS = {}

DATASET_FILE = acquisition.require_one_file(
    DATASET_FILE_SELECTOR
)
df = pd.read_csv(
    DATASET_FILE,
    **CSV_READ_OPTIONS,
)

print(f"Dataset file: {DATASET_FILE.name}")
print(f"Loaded shape: {df.shape}")


Dataset file: WA_Fn-UseC_-Telco-Customer-Churn.csv
Loaded shape: (7043, 21)


## 3. Unit of Observation

Each row represents one customer account identified by `customerID`.

The dataset records customer profile, subscribed services, contract and billing information, tenure, charges, and the `Churn` outcome at a specific observation point.

The uniqueness and completeness of `customerID` must be validated to confirm that each row represents one distinct customer account.

In [4]:
from scripts.validate_data import analyze_observation_unit


# Dataset-specific definition of the analytical observation.
OBSERVATION_ID = "customerID"
OBSERVATION_LABEL = "customer account"

# Produce a reusable completeness and uniqueness report.
observation_report = analyze_observation_unit(
    dataframe=df,
    identifier=OBSERVATION_ID,
)

print(f"Unit of observation: {OBSERVATION_LABEL}")
print(f"Observation identifier: {OBSERVATION_ID}")

observation_report.summary_frame()


Unit of observation: customer account
Observation identifier: customerID


,Metric,Value
0,Rows,7043
1,Non-null identifiers,7043
2,Unique identifiers,7043
3,Missing identifiers,0
4,Duplicate identifiers,0
5,Rows involved in duplication,0


In [5]:
# Confirm the expectations declared for this observation unit.
observation_report.raise_if_invalid(
    require_complete=True,
    require_unique=True,
)

print(
    "Validation passed: each row represents one distinct "
    f"{OBSERVATION_LABEL}."
)

if observation_report.has_duplicates:
    display(observation_report.duplicated_rows)
else:
    print("No duplicated observation identifiers were found.")


Validation passed: each row represents one distinct customer account.
No duplicated observation identifiers were found.


## 4. Dataset Structure

Summarize the number of rows and columns, the available fields, and the overall organization of the data.

In [6]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Rows: 7,043
Columns: 21


## 5. Feature and Target Identification

Identify the input features, the target variable, identifier fields, and columns that require special treatment.

In [7]:
# Column used as the prediction target.
TARGET_COLUMN = "Churn"

# Columns that identify observations but must not be used as predictors.
IDENTIFIER_COLUMNS = [
    "customerID",
]

In [8]:
# Validate whether all declared columns exist in the dataset.
declared_columns = {
    TARGET_COLUMN,
    *IDENTIFIER_COLUMNS,
}

missing_declared_columns = sorted(
    declared_columns.difference(df.columns)
)

if missing_declared_columns:
    raise KeyError(
        "Declared analytical columns were not found in the dataset: "
        f"{missing_declared_columns}"
    )

print("Declared analytical columns validated.")

Declared analytical columns validated.


In [9]:
# Candidate predictors exclude identifiers and the target variable.
EXCLUDED_FROM_FEATURES = {
    TARGET_COLUMN,
    *IDENTIFIER_COLUMNS,
}

FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in EXCLUDED_FROM_FEATURES
]

print(f"Target column: {TARGET_COLUMN}")
print(f"Identifier columns: {len(IDENTIFIER_COLUMNS)}")
print(f"Candidate features: {len(FEATURE_COLUMNS)}")

Target column: Churn
Identifier columns: 1
Candidate features: 19


In [11]:
import pandas as pd


def identify_column_role(column: str) -> str:
    """Return the analytical role assigned to a dataset column."""
    if column == TARGET_COLUMN:
        return "Target"

    if column in IDENTIFIER_COLUMNS:
        return "Identifier"

    return "Candidate feature"


column_roles = pd.DataFrame(
    {
        "Column": df.columns,
        "Role": [
            identify_column_role(column)
            for column in df.columns
        ],
        "Current dtype": [
            str(df[column].dtype)
            for column in df.columns
        ],
    }
)

column_roles

,Column,Role,Current dtype
0,customerID,Identifier,str
1,gender,Candidate feature,str
2,SeniorCitizen,Candidate feature,int64
3,Partner,Candidate feature,str
4,Dependents,Candidate feature,str
5,tenure,Candidate feature,int64
6,PhoneService,Candidate feature,str
7,MultipleLines,Candidate feature,str
8,InternetService,Candidate feature,str
9,OnlineSecurity,Candidate feature,str


## 6. Data Types

Review the expected and observed data types, highlighting columns that may have been loaded with an incorrect type.

## 7. Data Dictionary

Document the meaning, expected values, units, and analytical role of each relevant column.

## 8. Missing and Invalid Values

Investigate absent, blank, inconsistent, or invalid values and describe their potential impact on the analysis.

## 9. Duplicate Records

Check for duplicated observations and determine whether they represent data-quality issues or valid repeated records.

## 10. Target Distribution

Examine how the target classes or values are distributed and identify possible imbalance or unusual patterns.

## 11. Numerical Feature Exploration

Explore distributions, ranges, central tendencies, variability, outliers, and unusual values in numerical features.

## 12. Categorical Feature Exploration

Explore category frequencies, rare values, cardinality, inconsistent labels, and potentially relevant groupings.

## 13. Feature Relationships

Investigate associations between features to identify redundancy, dependencies, interactions, or unexpected relationships.

## 14. Feature-to-Target Relationships

Compare each relevant feature with the target to identify patterns, hypotheses, and variables that may support modeling.

## 15. Potential Data Leakage

Identify fields or transformations that may reveal the target directly or contain information unavailable at inference time.

## 16. Initial Data-Quality Findings

Consolidate the main structural and quality issues that must be addressed during data preparation.

## 17. Key Exploratory Insights

Summarize the most relevant patterns, contrasts, and hypotheses discovered during the exploratory analysis.

## 18. Preparation Decisions

Record the preliminary decisions that should guide cleaning, transformation, feature engineering, and dataset splitting.

## 19. Next Steps

List the actions that will be continued in the data-preparation and model-selection stages.